# Notebook for exploatory data analysis
Included checks:
- [] max and min transaction timestamps for fetched collections to make sure all records are properly extracted

## Imports & variable assignment

In [6]:
import pandas as pd
from datetime import datetime, timezone
import glob
import json
import os

In [7]:
opensea_preprocessed_data_folder = "data/processed/opensea/"

In [8]:
def load_parquet_folder(folder_path,):
    """
    Loads all Parquet files from a folder located one directory above
    the current working directory.
    """

    # Assemble base path (repo root)
    base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
    full_path = os.path.join(base_dir, folder_path)
    print(full_path)

    # Match all Parquet files
    parquet_files = glob.glob(os.path.join(full_path, "*.parquet"))
    print(f"✅ Found {len(parquet_files)} Parquet files in {folder_path}")

    if not parquet_files:
        return pd.DataFrame()

    dfs = []
    for file in parquet_files:
        df = pd.read_parquet(file)
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)
    print(
        f"📊 DataFrame created with {len(df)} rows and {len(df.columns)} columns "
        f"from {folder_path}"
    )

    return df

In [9]:
opensea_data_df = load_parquet_folder(folder_path=opensea_preprocessed_data_folder)
opensea_data_df.head()

/Users/juliaprzygoda/Desktop/thesis-code/nft-wash-trading-detection/data/processed/opensea/
✅ Found 3 Parquet files in data/processed/opensea/
📊 DataFrame created with 4399 rows and 15 columns from data/processed/opensea/


,event_timestamp,closing_date,transaction,chain,seller,buyer,nft.name,nft.contract,nft.collection,payment.quantity,payment.symbol,payment.decimals,payment.token_address,token_amount,event_day
0,2025-07-28 14:43:47,1753713827,0x9f763994020190b702b13f63c16448456c225e0c9013...,ethereum,0x7d533ac6cbcee97bf00b7ca10650eab713055736,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5418,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.170000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.70,2025-07-28
1,2025-07-28 11:23:11,1753701791,0x3269f0c515ea67d593aed3c6dc50767c660a45ec6e31...,ethereum,0x799224fdb4e635f5833d66e5ba61ffcd1c1fc558,0x978852db1bf809e30576c951733227d2a8fcc07d,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.733000e+19,ETH,18,0x0000000000000000000000000000000000000000,57.33,2025-07-28
2,2025-07-28 09:43:23,1753695803,0xed19e54a88fa195c9c75f22d4235a19353358ee37a8d...,ethereum,0x625202e9583039bd61cf599593d269b70521bfa1,0x24fedde1e5e2220256cb1819b7ee7f7b1b20ef5d,CryptoPunk #618,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.149000e+19,ETH,18,0x0000000000000000000000000000000000000000,51.49,2025-07-28
3,2025-07-28 08:47:47,1753692467,0x62cd1d149ad8ffbd9853a0df0b524a6a2bb2eb6cf0c4...,ethereum,0x4fc4457788f12aed1909cea1e4eeafaf2b8a4a00,0x8124ce96fbdce2cd3a80f0ba3b124a98a807be38,CryptoPunk #7364,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.850000e+19,ETH,18,0x0000000000000000000000000000000000000000,58.50,2025-07-28
4,2025-07-28 06:30:47,1753684247,0x342fc7ac6413262273136c4bba0e7a54243b32012253...,ethereum,0x86f39177283138fd6f5e344dfb78675ba4759ada,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,CryptoPunk #5512,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,cryptopunks,5.450000e+19,ETH,18,0x0000000000000000000000000000000000000000,54.50,2025-07-28


In [10]:
opensea_data_df.columns

Index(['event_timestamp', 'closing_date', 'transaction', 'chain', 'seller',
       'buyer', 'nft.name', 'nft.contract', 'nft.collection',
       'payment.quantity', 'payment.symbol', 'payment.decimals',
       'payment.token_address', 'token_amount', 'event_day'],
      dtype='str')

# Draft inital Wash Trading rules

## 1. Initial self-trades check

In [11]:
opensea_data_df["self_trade"] = (
    opensea_data_df["seller"] == opensea_data_df["buyer"]
)

self_trade_counts = opensea_data_df["self_trade"].value_counts()

print(self_trade_counts)

self_trade
False    4391
True        8
Name: count, dtype: int64


In [12]:
self_trades = opensea_data_df[opensea_data_df["self_trade"]]

if self_trades.empty:
    print("✅ No self-trades found")
else:
    print(f"🚨 Found {len(self_trades)} self-trades")
    display(
        self_trades[
            [
                "event_timestamp",
                "transaction",
                "seller",
                "buyer",
                "nft.name",
                "token_amount",
                "nft.collection",
            ]
        ]
    )

🚨 Found 8 self-trades


,event_timestamp,transaction,seller,buyer,nft.name,token_amount,nft.collection
935,2025-08-10 01:05:11,0xa4aea2116accd60f3f14846cba9aabe5eaa7a3953db4...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #7077,15.92000,pudgypenguins
1157,2025-08-14 13:36:23,0x3d970aa61501f3f9d19e2e36f5cd434b4d2542a90881...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,Pudgy Penguin #3681,13.43250,pudgypenguins
2169,2025-09-29 02:57:59,0xe95fb67ee5dee29935c092b3ff34ba139297a77d8af7...,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,0x3caebdddfb38d801cf4a2ec638085c36abea9c9e,Pudgy Penguin #1915,10.15740,pudgypenguins
3121,2025-09-11 18:25:59,0x09ecd6a6f589843389f526e46be3d87ee2019cb9c8e7...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#9494,9.20375,boredapeyachtclub
3250,2025-07-21 16:23:35,0x4a47f805d07a6bbe91683c65760d1707bf1c44fb7a89...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#1688,12.83550,boredapeyachtclub
3306,2025-08-02 13:48:11,0xc8b673ea6dc7fa237f1eed2f2b7396f573a6ac2b550b...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#640,12.08925,boredapeyachtclub
3682,2025-07-02 16:41:11,0xc85422fd4415de934241e2fd3179eb91dfc5da699ddc...,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,11.19375,boredapeyachtclub
4035,2025-08-18 12:11:35,0xfc35f2a18ab426d3784de61bb90c57cd870e2db3abcb...,0x54d28b26487e6529cf447434bccf439bea352e5d,0x54d28b26487e6529cf447434bccf439bea352e5d,#5457,12.62655,boredapeyachtclub


## 2. Holding period

In [13]:
df = opensea_data_df.sort_values("event_timestamp")

df["prev_sale_time"] = (
    df.groupby("nft.name")["event_timestamp"].shift(1)
)

df["holding_seconds"] = (
    df["event_timestamp"] - df["prev_sale_time"]
).dt.total_seconds()


THRESHOLD = 86_400  # 1 day

long_holds = df[df["holding_seconds"] > THRESHOLD]

print(f"Found {len(long_holds)} records above threshold")
display(
    long_holds[
        [
            "event_timestamp",
            "prev_sale_time",
            "holding_seconds",
            "seller",
            "buyer",
            "nft.name",
            "token_amount",
        ]
    ].head(20)
)

Found 1380 records above threshold


,event_timestamp,prev_sale_time,holding_seconds,seller,buyer,nft.name,token_amount
3686,2025-07-02 15:29:47,2025-07-01 14:30:59,89928.0,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x195f46025a6926968a1b3275822096eb12d97e70,#6209,10.440000
2366,2025-07-03 13:53:47,2025-07-02 12:18:11,92136.0,0x96e4a596087b811d61216684a4d87c688c6cbffa,0x6b8bc451c1d54ea4b67d2b3fdea9acd865f06eed,Pudgy Penguin #4691,9.840000
2363,2025-07-03 18:25:35,2025-07-02 04:52:23,135192.0,0xd5c4e4ed8486404df0b22a8f72a440e8d59fedba,0x8b291046edd260252fa00642f299144580ffe596,Pudgy Penguin #6612,9.970000
2362,2025-07-03 18:28:35,2025-07-01 16:47:35,178860.0,0xf2a7be2d334db950b8e490def5412827eac8494c,0x83735f0f1c732b28d044f22279a4f97218582dec,Pudgy Penguin #2897,9.740000
2360,2025-07-03 18:29:23,2025-07-02 05:58:59,131424.0,0x16f41a775a4ffc5da7b465597abfcb8ec0469d28,0x83735f0f1c732b28d044f22279a4f97218582dec,Pudgy Penguin #4792,9.699000
2359,2025-07-03 20:03:35,2025-07-02 00:59:11,155064.0,0xede4e6d73a5277910a6d5459d255e4d2095a2373,0xa6a8c4aff47e1a2e907cd1ef5161f14db7e0d9ba,Pudgy Penguin #3134,9.699990
3661,2025-07-04 16:24:23,2025-07-02 15:29:47,176076.0,0x195f46025a6926968a1b3275822096eb12d97e70,0xb2bb7767c46e385515c07ffaa68d1db79b3c845e,#6209,10.550000
3658,2025-07-05 06:07:11,2025-07-03 18:46:47,127224.0,0x6dcf1d3259c8a188e265ff72da5bd90f656df566,0xabe620ac5612535ced48f439f6d5232cf6e7ed88,#8997,11.290000
3654,2025-07-06 00:10:47,2025-07-01 03:02:35,421692.0,0x2777df13d272c9433858f5e1947b51e6a4720d60,0x53418545f4a97bb705eae9655213866d416207d5,#4733,11.777000
3652,2025-07-06 06:00:11,2025-07-03 19:30:11,210600.0,0x1ea27bce786a81022dfc156059771e8d3279a9a6,0x492da0e704a8e61fc08073e3b6cbafef4155593c,#9129,10.784103


In [14]:
ONE_HOUR = 60 * 60  # 3600 seconds

short_holds = df[df["holding_seconds"] <= ONE_HOUR]

print(f"🚨 Found {len(short_holds)} trades held ≤ 1 hour")

display(
    short_holds[
        [
            "event_timestamp",
            "prev_sale_time",
            "holding_seconds",
            "seller",
            "buyer",
            "nft.name",
            "token_amount",
        ]
    ].head(20)
)

🚨 Found 381 trades held ≤ 1 hour


,event_timestamp,prev_sale_time,holding_seconds,seller,buyer,nft.name,token_amount
2395,2025-07-01 13:51:47,2025-07-01 13:45:47,360.0,0x94e9d73a2aeec807d85756b84759965b41866450,0x621c70de47c35be4622c891555a6016cde2e1a61,Pudgy Penguin #2897,9.598956
3693,2025-07-01 21:57:47,2025-07-01 21:57:47,0.0,0x5129750f8c36b794e564d3fc5ef01c283d79ec36,0xfc02d4571ba182ac2fdf3e14fcb37f32e79bd28c,#7246,20.000000
3681,2025-07-02 16:41:11,2025-07-02 16:41:11,0.0,0x7e7022f8879d88bcc5d288b229737adb4b1f39cb,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,10.630000
3682,2025-07-02 16:41:11,2025-07-02 16:41:11,0.0,0xb751379ff5dbc7ae98279486f151172dd37abaa3,0xb751379ff5dbc7ae98279486f151172dd37abaa3,#4049,11.193750
3671,2025-07-03 15:49:47,2025-07-03 15:39:59,588.0,0x8bab603fde4824950fa81268a84ef9cbb89cad69,0x843fcb0cf448e714e735eb4b30e81186b9d58ec6,#524,10.740000
3667,2025-07-03 19:30:11,2025-07-03 18:40:47,2964.0,0xa6a8c4aff47e1a2e907cd1ef5161f14db7e0d9ba,0xf76246b0842c92ad5bd745973ca9eb85b937b126,#9129,10.430000
2350,2025-07-04 16:31:59,2025-07-04 16:30:11,108.0,0x1fee3385b22d69e93209db2042be58fcac57b59b,0xf2a7be2d334db950b8e490def5412827eac8494c,Pudgy Penguin #5156,9.300000
3896,2025-07-07 12:01:11,2025-07-07 12:00:23,48.0,0x8bab603fde4824950fa81268a84ef9cbb89cad69,0xf76246b0842c92ad5bd745973ca9eb85b937b126,#3465,11.540000
3895,2025-07-07 12:01:59,2025-07-07 12:01:11,48.0,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x66816bd969ff35688e9fd82b3e46fd505260216a,#3465,11.580000
2235,2025-07-07 23:20:59,2025-07-07 22:21:59,3540.0,0x2463a7b933a50188ff83ab5c703fe6ef3a0449c5,0xf76246b0842c92ad5bd745973ca9eb85b937b126,Pudgy Penguin #936,8.910000


## 3. Initial NFT flip count

In [15]:
df["nft_id"] = (
    opensea_data_df["nft.contract"] + "_" + opensea_data_df["nft.name"]
)

flip_counts = df.groupby("nft_id").size()
df["nft_flip_count"] = df["nft_id"].map(flip_counts)

# The ones fliped more often can be sus
# We need as well some time conditions
flip_distribution = df["nft_flip_count"].value_counts().sort_index()
print(flip_distribution)

nft_flip_count
1     869
2     970
3     693
4     548
5     340
6     348
7     112
8     112
9      99
10     70
11     22
12     24
13     26
14     28
17     17
22     22
99     99
Name: count, dtype: int64


## 4. Wallet-pair behavior (VERY important)
Same pair trades multiple times
Same NFT traded back and forth between same pair

In [16]:
pair_counts = (
    opensea_data_df.groupby(["seller", "buyer"])
      .size()
      .reset_index(name="pair_trade_count")
)

df = df.merge(pair_counts, on=["seller", "buyer"], how="left")
df.head()

# Threshold for “repeated trades” — 2 or more
REPEAT_THRESHOLD = 5

repeated_pairs = df[df["pair_trade_count"] >= REPEAT_THRESHOLD]

print(f"🚨 Found {len(repeated_pairs)} trades where the same seller/buyer pair traded ≥ {REPEAT_THRESHOLD} times")
display(
    repeated_pairs[
        [
            "event_timestamp",
            "seller",
            "buyer",
            "nft.name",
            "nft_id",
            "nft_flip_count",
            "pair_trade_count",
            "holding_seconds",
            "self_trade",
            "token_amount",
        ]
    ].head(20)
)

🚨 Found 76 trades where the same seller/buyer pair traded ≥ 5 times


,event_timestamp,seller,buyer,nft.name,nft_id,nft_flip_count,pair_trade_count,holding_seconds,self_trade,token_amount
20,2025-07-02 00:59:11,0xd8f24f5f0382e197c1e87ad82b357209383470cf,0xede4e6d73a5277910a6d5459d255e4d2095a2373,Pudgy Penguin #3134,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,6,9,26508.0,False,9.420100
44,2025-07-02 15:29:47,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x195f46025a6926968a1b3275822096eb12d97e70,#6209,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#6209,9,6,89928.0,False,10.440000
443,2025-07-12 11:40:23,0xd43dfa9024096b0224adf6859e22fbc35ea2b21f,0x3bb4fa84b120ac0dbb4a6bb0442fe2c47e324a93,Pudgy Penguin #1115,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,1,5,NaN,False,11.900000
540,2025-07-13 13:12:59,0xff53e1da7b67ae676d7742f858aab5bd4bc937f6,0x5b468edb7688e9ae6c1fa5a6d2debbef06e92907,#1049,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_#1049,6,14,960.0,False,11.579999
721,2025-07-15 00:23:35,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x195f46025a6926968a1b3275822096eb12d97e70,,0xbc4ca0eda7647a8ab7c2061c2e118a18a936f13d_,99,6,5772.0,False,11.490000
845,2025-07-16 18:20:47,0xd8f24f5f0382e197c1e87ad82b357209383470cf,0xede4e6d73a5277910a6d5459d255e4d2095a2373,Pudgy Penguin #6196,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,2,9,NaN,False,14.520100
958,2025-07-19 12:54:47,0xd43dfa9024096b0224adf6859e22fbc35ea2b21f,0x3bb4fa84b120ac0dbb4a6bb0442fe2c47e324a93,Pudgy Penguin #6732,0xbd3531da5cf5857e7cfaa92426877b022e612cf8_Pud...,4,5,509844.0,False,14.300000
1016,2025-07-20 18:52:35,0xa25803ab86a327786bb59395fc0164d826b98298,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,CryptoPunk #5918,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,1,7,NaN,False,47.999000
1030,2025-07-20 21:17:23,0x0232d1083e970f0c78f56202b9a666b526fa379f,0x8be240e8689547f1068a835d14f1d943958095dc,CryptoPunk #4398,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,1,13,NaN,False,47.000000
1034,2025-07-20 21:17:23,0x0232d1083e970f0c78f56202b9a666b526fa379f,0x8be240e8689547f1068a835d14f1d943958095dc,CryptoPunk #4656,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb_Cry...,1,13,NaN,False,47.000000


## Circular trading detection (A → B → A)

In [17]:
opensea_data_df["nft.id"] = opensea_data_df["nft.contract"] + "_" + opensea_data_df["nft.name"]

In [19]:
df = opensea_data_df
# Previous seller for the same NFT
df["prev_seller"] = df.groupby("nft.id")["seller"].shift(1)

# Did the NFT go back to the previous seller? (round-trip)
df["round_trip"] = df["buyer"] == df["prev_seller"]

In [21]:
round_trips = df[df["round_trip"]]

print(f"🚨 Found {len(round_trips)} round-trip trades")
display(
    round_trips[
        [
            "event_timestamp",
            "nft.name",
            "nft.contract",
            "seller",
            "buyer",
            "prev_seller",
            "token_amount",
            # "holding_seconds",
            "self_trade",
            # "pair_trade_count",
        ]
    ].head(20)
)

🚨 Found 1109 round-trip trades


,event_timestamp,nft.name,nft.contract,seller,buyer,prev_seller,token_amount,self_trade
17,2025-07-26 09:40:11,CryptoPunk #618,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0x017620d20e3c3fcd3d96ec3e1654f9cd94be88ed,0x625202e9583039bd61cf599593d269b70521bfa1,0x625202e9583039bd61cf599593d269b70521bfa1,48.26,False
34,2025-07-25 06:43:59,CryptoPunk #9370,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0x673a039f6a959fa9db65d16781e6defde30375d9,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,50.00,False
35,2025-07-25 06:41:59,CryptoPunk #8565,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0xbe006d5d3536f02c42a1ad815e3bf84f7fcb0e44,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,0x084263a11a1c2998ad4d1e5a79ee57d28ff714c2,50.00,False
45,2025-07-24 21:29:47,CryptoPunk #9001,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0x5baf7e5657cb76ee44d924ae36451c0ab60dee2a,0xff26ebc04cac6e19946afc8cf8ff46c7163d74db,0xff26ebc04cac6e19946afc8cf8ff46c7163d74db,49.95,False
57,2025-08-14 18:40:47,CryptoPunk #6741,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,50.00,False
59,2025-08-14 13:50:11,CryptoPunk #6741,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,0xf4c40bf7070fdcf64ecf020bcb583738a6cc3bcd,48.38,False
63,2025-08-14 10:54:59,CryptoPunk #6741,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0xfa823a137b8acbc4daeb365fdec7330e8d731088,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,0x1919db36ca2fa2e15f9000fd9cdc2edcf863e685,48.69,False
78,2025-08-12 23:34:11,CryptoPunk #6324,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0xde81549cff9f648b212b34b3311a6fcbc371bc05,0x90dcf53ae62df66e9f54861c5adac8a7b64f0842,0x90dcf53ae62df66e9f54861c5adac8a7b64f0842,53.00,False
96,2025-08-10 23:59:35,CryptoPunk #7190,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0xbb26a6da4d918682f58cc91bd3fb251dd28549d2,0x90dcf53ae62df66e9f54861c5adac8a7b64f0842,0x90dcf53ae62df66e9f54861c5adac8a7b64f0842,55.30,False
105,2025-08-24 16:10:35,CryptoPunk #3013,0xb47e3cd837ddf8e4c57f05d70ab865de6e193bbb,0x97df81b8bee956010124dc6d9c035360c3155455,0x34ae93aa8ba87d19e6bdc9e5beacfd0c78162345,0x34ae93aa8ba87d19e6bdc9e5beacfd0c78162345,49.50,False


## 5. Price abnormality (statistical signal)

In [ ]:
# daily_stats = (
#     df.groupby(["collection", "event_day"])
#       .agg(
#           median_price=("token_amount", "median"),
#           mad=("token_amount", lambda x: (x - x.median()).abs().median())
#       )
#       .reset_index()
# )

# df = df.merge(daily_stats, on=["collection", "event_day"], how="left")

# df["price_zscore"] = (
#     (df["token_amount"] - df["median_price"]) / df["mad"]
# )

## 6. Final wash trading score

In [ ]:
# df["wash_score"] = (
#     df["self_trade"].astype(int) * 5 +
#     (df["holding_seconds"] < 3600).astype(int) * 3 +
#     (df["nft_flip_count"] >= 3).astype(int) * 2 +
#     (df["pair_trade_count"] >= 2).astype(int) * 2 +
#     df["round_trip"].astype(int) * 3
# )